# CoralNet resize manifest

Explore the annotations parquet used by `CoralNetDataset`, join ETL `images` metadata
(`needs_resize`, dimensions, original `s3_key`), and build a new images manifest parquet
with resolved S3 URLs for training (original vs resized).

Related workflow (implemented on branch `claude/jovial-ptolemy-9ce9fb`):

- ETL: `uv run coralnet-etl all --upload-to-s3` → `coralnet_images_{RUN}.parquet`
- Resize: `uv run python scripts/run_coralnet_resize.py` → local/S3 checkpoint
- Manifest: this notebook (Ibis) → `coralnet_images_manifest_{RUN}.parquet`

Design reference: `docs/superpowers/specs/2026-05-25-coralnet-image-resizing-design.md`
(consolidated from stashes in commit `5fd8a0d`).

## 1. Setup

In [ ]:
import os

if "SM_CURRENT_HOST" not in os.environ:
    os.environ["AWS_PROFILE"] = "mermaid-core"

In [ ]:
from pathlib import Path

import hvplot.pandas  # noqa: F401 — registers .hvplot on pandas
import ibis
import pandas as pd
from great_tables import GT

from mermaidseg.datasets.coralnet.coralnet_dataset import (
    _LEGACY_ANNOTATIONS_PATH,
    _resolve_default_annotations_path,
)
from mermaidseg.datasets.coralnet.etl.config import (
    DEFAULT_BUCKET,
    DEFAULT_PREFIX,
    DEFAULT_RESIZE_THRESHOLD,
)
from mermaidseg.datasets.coralnet.preprocessing.manifest import build_manifest

ibis.options.interactive = True

In [ ]:
con = ibis.duckdb.connect()
con.raw_sql("""
    INSTALL httpfs;
    LOAD httpfs;
""")
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

In [ ]:
BUCKET = DEFAULT_BUCKET
SOURCE_PREFIX = DEFAULT_PREFIX
RUN = "20260526_807b611"
THRESHOLD = DEFAULT_RESIZE_THRESHOLD
RESIZE_OUTPUT_PREFIX = "dev/images"

# CoralNetDataset default annotations key (env vars override legacy literal)
DATASET_ANNOTATIONS_KEY = _resolve_default_annotations_path()
LEGACY_ANNOTATIONS_KEY = _LEGACY_ANNOTATIONS_PATH

ETL_BASE = f"s3://{BUCKET}/etl-outputs/coralnet/{RUN}"
ETL_ANNOTATIONS_URI = f"{ETL_BASE}/coralnet_annotations_{RUN}.parquet"
ETL_IMAGES_URI = f"{ETL_BASE}/coralnet_images_{RUN}.parquet"

DATASET_ANNOTATIONS_URI = f"s3://{BUCKET}/{DATASET_ANNOTATIONS_KEY}"
LEGACY_ANNOTATIONS_URI = f"s3://{BUCKET}/{LEGACY_ANNOTATIONS_KEY}"

# Local resize checkpoint from scripts/run_coralnet_resize.py (override if pulled from S3)
CHECKPOINT_PATH = Path("outputs/resize_full_20260526_807b611.parquet")
CHECKPOINT_S3_URI = f"s3://{BUCKET}/etl-outputs/coralnet/{RUN}/resize_checkpoint_{RUN}.parquet"

OUTPUT_MANIFEST_LOCAL = Path(f"outputs/coralnet_images_manifest_{RUN}.parquet")
OUTPUT_MANIFEST_S3_KEY = f"etl-outputs/coralnet/{RUN}/coralnet_images_manifest_{RUN}.parquet"

## 2. Explore current CoralNetDataset annotations parquet

In [ ]:
dataset_annotations = con.read_parquet(DATASET_ANNOTATIONS_URI)
legacy_annotations = con.read_parquet(LEGACY_ANNOTATIONS_URI)
etl_annotations = con.read_parquet(ETL_ANNOTATIONS_URI)

In [ ]:
annotation_paths = pd.DataFrame(
    {
        "label": [
            "CoralNetDataset default",
            "Legacy literal",
            f"ETL run {RUN}",
        ],
        "s3_uri": [DATASET_ANNOTATIONS_URI, LEGACY_ANNOTATIONS_URI, ETL_ANNOTATIONS_URI],
        "rows": [
            dataset_annotations.count().execute(),
            legacy_annotations.count().execute(),
            etl_annotations.count().execute(),
        ],
        "sources": [
            dataset_annotations.source_id.nunique().execute(),
            legacy_annotations.source_id.nunique().execute(),
            etl_annotations.source_id.nunique().execute(),
        ],
        "unique_images": [
            dataset_annotations.select("source_id", "image_id").distinct().count().execute(),
            legacy_annotations.select("source_id", "image_id").distinct().count().execute(),
            etl_annotations.select("source_id", "image_id").distinct().count().execute(),
        ],
    }
)

In [ ]:
GT(annotation_paths).tab_header(
    title="Annotations parquet comparison",
    subtitle=f"CoralNetDataset resolves to `{DATASET_ANNOTATIONS_KEY}`",
)

In [ ]:
etl_status_counts = (
    etl_annotations.group_by("status")
    .aggregate(n=etl_annotations.count())
    .order_by(ibis.desc("n"))
    .execute()
)

In [ ]:
GT(etl_status_counts).tab_header(title=f"ETL annotation rows by status ({RUN})")

## 3. Load ETL images + resize checkpoint

In [ ]:
images = con.read_parquet(ETL_IMAGES_URI)

In [ ]:
if CHECKPOINT_PATH.exists():
    checkpoint = con.read_parquet(str(CHECKPOINT_PATH))
    checkpoint_source = str(CHECKPOINT_PATH)
else:
    checkpoint = con.read_parquet(CHECKPOINT_S3_URI)
    checkpoint_source = CHECKPOINT_S3_URI

In [ ]:
resize_summary = pd.DataFrame(
    {
        "Metric": [
            "ETL images rows",
            "Images with needs_resize=True",
            "Checkpoint rows",
            "Checkpoint completed",
            "Checkpoint pending",
            "Checkpoint failed",
        ],
        "Value": [
            images.count().execute(),
            images.filter(images.needs_resize).count().execute(),
            checkpoint.count().execute(),
            checkpoint.filter(checkpoint.status == "completed").count().execute(),
            checkpoint.filter(checkpoint.status == "pending").count().execute(),
            checkpoint.filter(checkpoint.status == "failed").count().execute(),
        ],
    }
)
resize_summary["checkpoint_source"] = checkpoint_source

In [ ]:
GT(resize_summary[["Metric", "Value"]]).tab_header(
    title="Images vs resize checkpoint",
    subtitle=checkpoint_source,
)

## 4. Build manifest with Ibis (resolved S3 URLs)

For each image in the ETL `images` parquet:

- `original_s3_key` — from ETL `s3_key` (under `coralnet-public-images/...`)
- `resized_s3_key` — `{RESIZE_OUTPUT_PREFIX}/resized/s{source_id}/images/{image_id}.jpg`
  (matches `preprocessing/resize.py`)
- `image_s3_key` — resized key when `needs_resize` and checkpoint `status='completed'`,
  otherwise the original key
- `image_s3_uri` — `s3://{bucket}/{image_s3_key}` for downstream loaders

In [ ]:
joined = images.left_join(checkpoint, ["source_id", "image_id"])

resized_s3_key = (
    ibis.literal(f"{RESIZE_OUTPUT_PREFIX}/resized/s")
    + joined.source_id.cast("string")
    + "/images/"
    + joined.image_id
    + ".jpg"
)

use_resized = joined.needs_resize & (joined.status == "completed")

scale = ibis.literal(THRESHOLD) / joined.longest_edge.cast("float64")
resized_width = ibis.ifelse(
    joined.needs_resize & joined.longest_edge.notnull() & (joined.longest_edge > THRESHOLD),  # noqa: PD004 — ibis API, not pandas
    (joined.width.cast("float64") * scale).cast("int32"),
    joined.width,
)
resized_height = ibis.ifelse(
    joined.needs_resize & joined.longest_edge.notnull() & (joined.longest_edge > THRESHOLD),  # noqa: PD004 — ibis API, not pandas
    (joined.height.cast("float64") * scale).cast("int32"),
    joined.height,
)

manifest_ibis = joined.mutate(
    original_s3_key=joined.s3_key,
    resized_s3_key=resized_s3_key,
    image_s3_key=ibis.ifelse(use_resized, resized_s3_key, joined.s3_key),
    image_s3_uri=ibis.literal(f"s3://{BUCKET}/")
    + ibis.ifelse(use_resized, resized_s3_key, joined.s3_key),
    resize_status=joined.status,
    load_width=ibis.ifelse(use_resized, resized_width, joined.width),
    load_height=ibis.ifelse(use_resized, resized_height, joined.height),
    uses_resized_image=use_resized,
)

In [ ]:
manifest_summary = pd.DataFrame(
    {
        "Metric": [
            "Total images",
            "Using resized S3 key",
            "Using original S3 key",
            "needs_resize but not yet completed",
            "needs_resize and failed",
        ],
        "Value": [
            manifest_ibis.count().execute(),
            manifest_ibis.filter(manifest_ibis.uses_resized_image).count().execute(),
            manifest_ibis.filter(~manifest_ibis.uses_resized_image).count().execute(),
            manifest_ibis.filter(
                manifest_ibis.needs_resize
                & manifest_ibis.resize_status.fill_null("pending").eq("pending")
            )
            .count()
            .execute(),
            manifest_ibis.filter(
                manifest_ibis.needs_resize & (manifest_ibis.resize_status == "failed")
            )
            .count()
            .execute(),
        ],
    }
)

In [ ]:
GT(manifest_summary).tab_header(title="Resolved image S3 key summary")

In [ ]:
manifest_sample = (
    manifest_ibis.filter(manifest_ibis.needs_resize)
    .select(
        "source_id",
        "image_id",
        "needs_resize",
        "resize_status",
        "uses_resized_image",
        "original_s3_key",
        "resized_s3_key",
        "image_s3_uri",
        "width",
        "height",
        "load_width",
        "load_height",
    )
    .limit(10)
    .execute()
)

In [ ]:
GT(manifest_sample).tab_header(title="Sample rows (needs_resize=True)")

## 5. Cross-check with `build_manifest` (ibis)

The preprocessing module builds a resize-only manifest from checkpoint rows.
Compare counts to sanity-check the Ibis join.

In [ ]:
df_resize_manifest = build_manifest(
    images=images,
    checkpoint=checkpoint,
    output_prefix=RESIZE_OUTPUT_PREFIX,
    threshold=THRESHOLD,
).to_pandas()

pandas_manifest_summary = pd.DataFrame(
    {
        "source": ["build_manifest (resize rows only)", "Ibis (all images)"],
        "rows": [len(df_resize_manifest), manifest_ibis.count().execute()],
        "completed": [
            (df_resize_manifest["status"] == "completed").sum(),
            manifest_ibis.filter(manifest_ibis.uses_resized_image).count().execute(),
        ],
    }
)

In [ ]:
GT(pandas_manifest_summary).tab_header(title="Ibis vs build_manifest row counts")

## 6. Write manifest parquet

In [ ]:
manifest_cols = [
    "source_id",
    "image_id",
    "original_s3_key",
    "resized_s3_key",
    "image_s3_key",
    "image_s3_uri",
    "needs_resize",
    "uses_resized_image",
    "resize_status",
    "width",
    "height",
    "longest_edge",
    "load_width",
    "load_height",
    "header_status",
    "error_message",
]

manifest_df = manifest_ibis.select(*manifest_cols).order_by("source_id", "image_id").execute()

OUTPUT_MANIFEST_LOCAL.parent.mkdir(parents=True, exist_ok=True)
manifest_df.to_parquet(OUTPUT_MANIFEST_LOCAL, index=False)

In [ ]:
write_summary = pd.DataFrame(
    {
        "artifact": ["local manifest", "S3 upload target"],
        "path": [str(OUTPUT_MANIFEST_LOCAL.resolve()), f"s3://{BUCKET}/{OUTPUT_MANIFEST_S3_KEY}"],
        "rows": [len(manifest_df), len(manifest_df)],
    }
)

GT(write_summary).tab_header(title="Manifest written")

## 7. Visualizations

In [ ]:
needs_resize_manifest = manifest_ibis.filter(manifest_ibis.needs_resize).mutate(
    resize_status=manifest_ibis.resize_status.fill_null("not_in_checkpoint")
)

resize_status_counts = (
    needs_resize_manifest.group_by("resize_status")
    .aggregate(n=needs_resize_manifest.count())
    .order_by(ibis.desc("n"))
    .execute()
)

In [ ]:
resize_status_counts.hvplot.bar(
    x="resize_status",
    y="n",
    title="needs_resize images by checkpoint status",
    xlabel="Resize status",
    ylabel="Images",
    rot=45,
    height=400,
    width=700,
)

In [ ]:
needs_resize_df = manifest_df.loc[manifest_df["needs_resize"]]
sample_n = min(5000, len(needs_resize_df))
load_dims_df = needs_resize_df.sample(n=sample_n, random_state=42)

In [ ]:
load_dims_df.hvplot.scatter(
    x="width",
    y="height",
    by="uses_resized_image",
    title="Original dimensions (needs_resize sample)",
    xlabel="Width (px)",
    ylabel="Height (px)",
    alpha=0.4,
    height=400,
    width=700,
    legend="top_right",
)

## 8. Optional: upload manifest to S3

Uncomment to publish after review.

In [ ]:
# import boto3
#
# boto3.client("s3").upload_file(
#     str(OUTPUT_MANIFEST_LOCAL),
#     BUCKET,
#     OUTPUT_MANIFEST_S3_KEY,
# )